In [0]:
SELECT
    source_month,
    COUNT(*) AS bronze_rows,
    COUNT(DISTINCT _source_file) AS source_files
FROM nyc_taxi_v2.bronze.yellow_trips
GROUP BY source_month
ORDER BY source_month;


SELECT
    source_month,
    rows_received,
    rows_passed,
    rows_quarantined,
    rows_warned,
    rows_received = rows_passed + rows_quarantined
        AS reconciles
FROM nyc_taxi_v2.gold.data_quality_summary
WHERE rule = 'ALL'
ORDER BY source_month;


WITH accepted AS (
    SELECT source_month, COUNT(*) AS accepted_rows
    FROM nyc_taxi_v2.silver.yellow_trips
    GROUP BY source_month
),
rejected AS (
    SELECT source_month, COUNT(*) AS rejected_rows
    FROM nyc_taxi_v2.silver.trip_quarantine
    GROUP BY source_month
)
SELECT
    d.source_month,
    d.rows_passed,
    COALESCE(a.accepted_rows, 0) AS actual_accepted,
    d.rows_quarantined,
    COALESCE(q.rejected_rows, 0) AS actual_rejected
FROM nyc_taxi_v2.gold.data_quality_summary d
LEFT JOIN accepted a USING (source_month)
LEFT JOIN rejected q USING (source_month)
WHERE d.rule = 'ALL'
ORDER BY d.source_month;


SELECT trip_key, COUNT(*) AS occurrences
FROM nyc_taxi_v2.silver.yellow_trips
GROUP BY trip_key
HAVING COUNT(*) > 1;


SELECT
    source_month,
    trip_key,
    failed_rules,
    warning_rules,
    _source_file,
    _raw_record
FROM nyc_taxi_v2.silver.trip_quarantine
LIMIT 100;

SELECT
    (SELECT COUNT(*)
     FROM nyc_taxi_v2.silver.yellow_trips) AS silver_trips,

    (SELECT COALESCE(SUM(trips), 0)
     FROM nyc_taxi_v2.gold.daily_zone_revenue) AS daily_trips,

    (SELECT COALESCE(SUM(trip_count), 0)
     FROM nyc_taxi_v2.gold.hourly_demand) AS hourly_trips;


SELECT
    g.pickup_date,
    z.borough,
    z.zone,
    g.trips,
    g.total_fare,
    g.total_tips,
    g.average_distance
FROM nyc_taxi_v2.gold.daily_zone_revenue g
LEFT JOIN nyc_taxi_v2.silver.taxi_zones z
    ON g.pu_location_id = z.location_id
ORDER BY g.pickup_date, g.total_fare DESC;